# Track B â€” dual-view 1D-CNN classifier
### Astronet / Yu et al. (2019) style, for PS7 O4â€“O5

The tabular LightGBM (`classify.py`) reads engineered vetting *features*. This CNN reads the **shape** of the phase-folded transit directly, via two views:

- **global view** â€” whole phase-folded curve (`N_GLOBAL=201` bins): period, secondary eclipses, out-of-transit variability.
- **local view** â€” zoom on the transit (`N_LOCAL=61` bins): ingress/egress slope, U-vs-V shape, depth.

Two 1D-conv branches â†’ concat â†’ dense â†’ softmax over transit / EB / blend / other. Finally we **late-fuse** the CNN with the LightGBM (averaged probabilities) â€” typically stronger than either alone.

In [ ]:
# --- Setup -------------------------------------------------------------------------
# LOCAL: imports `exopipeline` from the project root. COLAB: upload exopipeline/ then
# `pip install torch` (Colab has it preinstalled) + the other requirements.
import sys, os
for p in [".", "..", r"g:\Exoplanet project"]:
    if os.path.isdir(os.path.join(p, "exopipeline")):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from exopipeline import labels, cnn, classify, config
print("ready")


## 1. Build the dual-view dataset from the known TOIs
For each labelled target: ingest â†’ detrend â†’ focused search at the known period (for `t0`/duration) â†’ phase-fold into the global & local views. Cached to `data/features/cnn_views.npz` so re-runs are instant.

In [ ]:
if cnn.VIEW_DATASET.exists():
    Xg, Xl, y, tics = cnn.load_view_dataset()
else:
    # CLEAN labels: TESS EB catalog + confirmed planets (see README ablation)
    sample = labels.build_clean_sample(per_class=80)
    Xg, Xl, y, tics = cnn.build_view_dataset(sample, max_sectors=4)
print("views:", Xg.shape, Xl.shape, "| classes:", dict(zip(*np.unique(y, return_counts=True))))

### Sanity â€” what the network sees
Average global/local view per class. Transits show a clean centred U; EBs a deeper / V-shaped dip, often with a secondary; 'other' is flat.

In [ ]:
classes = sorted(np.unique(y))
fig, ax = plt.subplots(2, len(classes), figsize=(3*len(classes), 5), squeeze=False)
for j, cl in enumerate(classes):
    sel = y == cl
    ax[0, j].plot(Xg[sel].mean(0), lw=1); ax[0, j].set_title(f"{cl}\nglobal (n={sel.sum()})")
    ax[1, j].plot(Xl[sel].mean(0), lw=1, color="C1"); ax[1, j].set_title("local")
plt.tight_layout(); plt.show()

## 2. Train the dual-view CNN
Class-weighted cross-entropy handles the imbalance. Small model, CPU-friendly.

In [ ]:
res = cnn.train_cnn(Xg, Xl, y, n_epochs=60, batch_size=32, lr=1e-3)
cm = pd.DataFrame(res["confusion_matrix"], index=res["classes"], columns=res["classes"])
print("Confusion matrix (rows=true, cols=pred):"); print(cm)
rep = pd.DataFrame(res["report"]).T
rep[["precision","recall","f1-score"]].round(3)

## 3. Late-fusion ensemble (CNN + LightGBM)
`cnn.predict_ensemble(features, global_view, local_view)` averages the calibrated tabular probabilities with the CNN's. Demonstrate on one target end-to-end.

In [ ]:
from exopipeline import ingest, detrend, search, vetting
star = ingest.clean(ingest.load_star(config.DEMO_BLIND, max_sectors=4))  # TOI-270
flat = detrend.detrend(star.time, star.flux)
cand = search.find_planets(flat.time, flat.flux, max_planets=1, refine=False, verbose=False)[0]
feats = vetting.compute_features(flat.time, flat.flux, cand, crowdsap=star.crowdsap)
g, l = cnn.make_views(flat.time, flat.flux, cand.period, cand.T0, cand.duration)
print("LightGBM :", classify.predict(feats))
print("CNN      :", cnn.predict_cnn(g, l))
print("Ensemble :", cnn.predict_ensemble(feats, g, l, w_cnn=0.5))

---
**Summary.** A dual-view 1D-CNN trained on the known TOIs classifies transit / EB / blend / other directly from phase-folded shape, and late-fuses with the calibrated LightGBM for a stronger combined verdict (O4/O5). The ensemble is the model the vetting sheet and Streamlit app can call for the final confidence.